# Detrending climate timeseries in the Cal-Adapt Analytics Engine
This notebook covers how to find a remove a simple trend from a temperature projection
1. using simple linear regression
2. fitting a higher-order polynomial
3. using an ensemble mean to remove only that part of the trend which is "forced" by global warming, as opposed to resulting from climate variability
4. and taking into account the seasonal and diurnal cycles in hourly data

## Step 0: Setup
This cell imports the python library [climakitae](https://github.com/cal-adapt/climakitae), our AE toolkit for climate data analysis, and a few other things we'll need here.

In [ ]:
import climakitae as ck
import statsmodels.api as sm
import numpy as np
import matplotlib.pyplot as plt

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

## Step 1: Select data
Today we will work with some of the data from LOCA using SSP 5-8.5.

**What to select:** select 'Statistical' only, 'SSP 3-7.0', stick with temperature in your preferred units, choose a single county, and select 'Yes' to 'Compute an area average...'.

To learn more about the data available on the Analytics Engine, [see our data catalog](https://analytics.cal-adapt.org/data/). 

In [ ]:
app.select()

## Step 2: Retrieve data
Call app.retrieve(), to assign the subset/combo of data specified to a variable name of your choosing, in an xarray [DataArray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.html) format.

In [ ]:
data_to_use = app.retrieve()

In [ ]:
data_to_use

For this first part, we'll work with annual averaged data

In [ ]:
data_to_use = data_to_use.resample(time='AS-SEP').mean()

Now load, and then take an inital view of it.

[*warning:* this load step can take ~13min currently on the hub]

In [ ]:
%%time
data_to_use = app.load(data_to_use)

In [ ]:
app.view(data_to_use)

## Step 3: Detrending
### Method 1: subtract a linear regression

First a quick little function to fit a linear regression and subtract it from the original data. 

In [ ]:
def linear_detrend(y):
    y = y.squeeze()
    X = range(0,len(y.time))
    X = sm.add_constant(X)
    results = sm.OLS(y.values, X).fit()
    y.data = y.values - results.fittedvalues
    return y 
    

Now apply that over all of the simulations in data_to_use.

In [ ]:
detrend1 = data_to_use.squeeze().groupby('simulation').apply(linear_detrend)

And view the detrended data, just as you viewed the original data.

In [ ]:
app.view(detrend1)

### Method 2: subtract a higher order polynomial

For this we will modify the function in Method 1, to fit and subtract a forth order polynomial. This can be a better fit that a simple line for temperature data over the entire 21st century under a high greenhouse-gas scenario.

In [ ]:
def forth_order_fit(y):
    x = np.arange(0,len(y.time))
    X = np.column_stack((x, x**2, x**3, x**4))
    X = sm.add_constant(X)
    results = sm.OLS(y.values, X).fit()
    return results.fittedvalues

def forth_order_detrend(y):
    y = y.squeeze()
    y.data = y.values - forth_order_fit(y)
    return y 
       

In [ ]:
detrend2 = data_to_use.groupby('simulation').map(forth_order_detrend)

In [ ]:
app.view(detrend2)

### Method 3: subtract an ensemble mean

It's fine to remove the trend as shown above, but sometimes we may wish to distinguish and remove only the component of the trend that is "forced" by the radiative imbalance caused by greenhouse gasses in the atmosphere. If we are fortunate enough to have many simulations from the same model we can approximate this with an ensemble mean. This ensemble mean is an average across a suite of simulations that were run with a single model such that only the initial conditions were altered in each simulation. 

The set of simulations in this type of single-model ensemble has an identical response to climate change, but with different timing and phasing of variability in a given year or decade. The ensemble mean averages-away the variability, leaving only the forced response of climate change. 

Because we have only 10 or fewer ensemble members from each downscaled GCM, there would still be too much variability in the ensemble mean so we will still fit a function to it to represent the forced trend. 

Turning around and subtracting the ensemble mean fitted-trend from each individual simulation leaves only the variability. 

The way the detrending function below is written, if there is only one ensemble member available, the result will be the same as method 2 above.

In [ ]:
def model_id(y):
    return str(y.values).split('_')[1]

def ensemble_mean_detrend(y):
    '''
    Assumes a dataarray returned by app.retrieve() for 'Statistical' only. 
    
    This method would not be relevant for dynamically-downscaled data for 
    which only one ensemble member is available per GCM. And this is not 
    currently written to check-for and exclude unanticipated inputs.
    '''
    model_ids = list(set([model_id(i) for i in y.simulation]))
    for model in model_ids:
        sim_list = [i for i in y.simulation.values if model in i]
        subset = y.sel(simulation=sim_list)
        y.loc[dict(simulation=sim_list)] = subset - forth_order_fit(subset.mean('simulation'))
    return y

In [ ]:
detrend3 = ensemble_mean_detrend(data_to_use.squeeze())

In [ ]:
app.view(detrend3)

### Method 4: Account for the seasonal and diurnal cycles
Up to now we've worked with annual average timeseries. Now we'll load some different data that has much higher time-resolution to understand some of the challenges of dealing with nonstationary timeseries.

#### Load hourly data for this one:
**What to select:** In the 'select' panel, choose 'dynamical' this time, with hourly data at 9km. Try with SSP 5-8.5 so that there is only one simulation, and don't forget to select 'yes' to 'Compute an area average...' for some small county, as before.

In [ ]:
app.select()

In [ ]:
data_hourly = app.retrieve()

In [ ]:
data_hourly = app.load(data_hourly.squeeze())

In [ ]:
app.view(data_hourly)

And let's examine a bit more about this data. We'll need some helper functions to view:
1. seasonal cycle
2. diurnal cycle
3. a probability density function
    
For each of these we'll want to see a historical period (1981-2010) and a future period (final 30 years).

In [ ]:
def extract_periods(data):
    data_hist = data.sel(time=slice('1981','2010'))
    # last 30 years:
    data_fut = data.isel(time=slice(-30*365*24,-1))
    return data_hist, data_fut

def seasonal_cycle(data_hist,data_fut):
    data_hist.groupby('time.month').mean('time').plot()
    data_fut.groupby('time.month').mean('time').plot()
    plt.legend(['Historical','Future'])
    plt.title('Seasonal Cycle')

def diurnal_cycle(data_hist,data_fut):
    data_hist.groupby('time.hour').mean('time').plot()
    data_fut.groupby('time.hour').mean('time').plot()
    plt.legend(['Historical','Future'])
    plt.title('Diurnal Cycle')

def pdf_plot(data):
    kde = sm.nonparametric.KDEUnivariate(data)
    kde.fit()
    plt.plot(kde.support,kde.density)
    
def pdf_overview(data_hist,data_fut):
    pdf_plot(data_hist)
    pdf_plot(data_fut)
    plt.legend(['Historical','Future'])
    plt.title('PDF')

def summary_plots(data):
    data_hist, data_fut = extract_periods(data)
    plt.figure(figsize=[15,5])
    plt.subplot(131)
    seasonal_cycle(data_hist,data_fut)
    plt.subplot(132)
    diurnal_cycle(data_hist,data_fut)
    plt.subplot(133)
    pdf_overview(data_hist,data_fut)

Now use those functions to make plots.

In [ ]:
summary_plots(data_hourly)

#### Now, the detrending procedure
First we'll try treating each month and time of day separately to see why this doesn't work.

Because there is only a single timeseries to begin with, we don't need to group by simulation, and can jump directly to grouping by month, and then time of day.

In [ ]:
def month_to_time(y):
    return y.groupby('time.hour').map(linear_detrend)

In [ ]:
detrend_by_month_time = data_hourly.groupby('time.month').map(month_to_time)

In [ ]:
app.view(detrend_by_month_time)

While this results detrend_by_month_timein data that matches the historical PDF, the seasonal and diurnal cycles are not preserved.

In [ ]:
summary_plots(detrend_by_month_time)

To get around this, we would need to block-remove the seasonal and diurnal cycles, detrend, and then add back in the seasonal and diurnal cycles of the present. (Not shown.)

Next we'll show how doing a simple detrending for the entire century is also not ideal for producing additional samples of current variability.

In [ ]:
detrend_simple = linear_detrend(data_hourly)

In [ ]:
app.view(detrend_simple)

The PDF for the future period (30 years at end-of-century) is not the same as the historical.

In [ ]:
summary_plots(detrend_simple)

But, if we consider only a period centered around the present...

In [ ]:
summary_plots(detrend_simple.sel(time=slice('1980','2038')))

Compare the 'future' behavior with that before detrending:

In [ ]:
summary_plots(data_hourly.sel(time=slice('1980','2038')))

This suggests that the 'moving-window' approach to using detrending to increase sampling is reasonable even with a simple detrending method, and more complicated de-composition and re-composition of the timeseries is not needed.